# RAG Script for basic architecture Understanding.

## Installs
Note : if you are doing it in a python project based env. use a virtual environment.


In [1]:
! pip install google-generativeai pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

## imports

In [2]:
import google.generativeai as genai
from pypdf import PdfReader

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
def process_pdf(file_path, chunk_size=1000, overlap=200):
    print(f"Reading PDF: {file_path}")
    reader = PdfReader(file_path)

    # Extract raw text
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    # Chunking logic (Sliding Window)
    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunk = full_text[start:end]
        chunks.append(chunk)
        start += (chunk_size - overlap)

    return chunks

## test process pdf

In [4]:
chunks = process_pdf("Slide 05 - RAG I.pdf")
print(chunks)

Reading PDF: Slide 05 - RAG I.pdf
['Retrieval Augmented \nGeneration\nDS205.3 – Data Science in Python\nMr. Anton Jayakody\nMSc (CS)[R], BSc (CS)\nSo far:\n• Week 1: Data engineering basics (asyncio, type hinting).\n• Week 2: Agentic architecture (OOP , composition, and \ndependency injection).\n• Week 3: Tool routing (The Router pattern to select the right \ncapability).\n• Week 4: Prompt Engineering\nDS205.3 2\nDrawbacks of LLMs\n• Hallucination \n• Outdated information \n• Low efficiency in parameterizing knowledge \n• Lack of in-depth knowledge in specialized domains \n• Weak inferential capabilities\nDS205.3 3\nAugmented Generation\nDS205.3 4\nAdvantages of Augmented Generation\nDS205.3 5\n• Domain-specific accurate answering \n• Frequent updates of data \n• Traceability and explainability of generated content \n• Controllable Cost \n• Privacy protection of data\nRetrieval-Augmented Generation (RAG)\nDS205.3 6\n• When answering questions or generating text, it first retrieves rele

# Creating the Chroma DB

In [5]:
import chromadb
from chromadb.utils import embedding_functions

In [6]:
# 2. Setup the Embedding Function (Local Sentence-Transformers)
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 3. Initialize ChromaDB (Persistent storage in a local folder)
db_client = chromadb.PersistentClient(path="./chroma_db_notebook")
collection = db_client.get_or_create_collection(
    name="lecture_notes",
    embedding_function=embedding_fn
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Embedding the chunks to the DB

In [12]:
def chunks_to_db(chunks):
  ids = [f"id_{i}" for i in range(len(chunks))]
  collection.add(documents=chunks, ids=ids)

  print(f"Successfully indexed {len(chunks)} chunks")

In [13]:
# chunks = process_pdf("Slide 05 - RAG I.pdf")
chunks_to_db(chunks)

Successfully indexed 3 chunks


# The Inference

In [14]:
def trace_rag_process(user_query):
    print("=== [1] INPUT PROMPT ===")
    print(f"User Question: '{user_query}'\n")

    # --- [2] EMBEDDING & RETRIEVAL ---
    results = collection.query(query_texts=[user_query], n_results=2)
    retrieved_chunks = results['documents'][0]

    print("=== [2] RETRIEVED CHUNKS (Top 2 matches) ===")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"--- Chunk {i+1} (Source: ChromaDB) ---")
        print(f"{chunk.strip()[:300]}...")
    print("\n")

    # --- [3] FINAL AUGMENTED PROMPT ---
    context_str = "\n---\n".join(retrieved_chunks)
    final_prompt = f"""
    SYSTEM INSTRUCTION: Answer based ONLY on the context.

    CONTEXT:
    {context_str}

    USER QUESTION:
    {user_query}
    """

    print("=== [3] FINAL AUGMENTED PROMPT (What Gemini actually sees) ===")
    print(final_prompt)
    print("-" * 50 + "\n")

    # --- [4] FINAL OUTPUT ---
    genai.configure(api_key="YOUR_API_KEY")
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(final_prompt)

    print("=== [4] FINAL OUTPUT (The Generation) ===")
    print(response.text)


In [15]:
user_input = input()
trace_rag_process(user_input)

What is Rag?
=== [1] INPUT PROMPT ===
User Question: 'What is Rag?'

=== [2] RETRIEVED CHUNKS (Top 2 matches) ===
--- Chunk 1 (Source: ChromaDB) ---
n of these 
to a large language model to generate an answer
Basic RAG / Naïve RAG
DS205.3 8
Step1 Indexing 
• 1. Divide the document into even chunks, each chunk being a piece of 
the original text. 
• 2. Using the encoding model to generate an embedding for each 
chunck. 
• 3. Store the Embedding o...
--- Chunk 2 (Source: ChromaDB) ---
tection of data
Retrieval-Augmented Generation (RAG)
DS205.3 6
• When answering questions or generating text, it first retrieves relevant 
information from a large number of documents, and then LLMs generates 
answers based on this information. 
• By attaching a external knowledge base, there is no ...


=== [3] FINAL AUGMENTED PROMPT (What Gemini actually sees) ===

    SYSTEM INSTRUCTION: Answer based ONLY on the context.

    CONTEXT:
    n of these 
to a large language model to generate an answer
Basic 

DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.